**Install Streamlit & Pyngrok**

In [12]:
!pip -q install streamlit pyngrok faiss-cpu sentence-transformers groq

**Tambahkan API -> Groq & Ngrok**

In [13]:
import os
from google.colab import userdata
from pyngrok import ngrok

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
NGROK_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)
print("Groq dan ngrok secrets berhasil dimuat.")

Groq dan ngrok secrets berhasil dimuat.


**Running Streamlit & Open Tunnel**

In [14]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

In [15]:
from pathlib import Path
app_code = 'import os, re, json, hashlib\nfrom pathlib import Path\nimport numpy as np\nimport streamlit as st\nimport faiss\nfrom sentence_transformers import SentenceTransformer\nfrom groq import Groq\n\nMODEL_NAME = "qwen/qwen3.8-27b"\nMEMORY_FILE = "/content/wisata_kalsel_chat_memory.json"\nst.set_page_config(page_title="Wisata Kalsel AI", page_icon="🌿", layout="wide")\n\nRAG_DATA = [\n {"title":"Pasar Terapung Lok Baintan","category":"destinasi, budaya Banjar","text":"Pasar terapung tradisional di Desa Sungai Pinang, Kecamatan Sungai Tabuk, Kabupaten Banjar. Perdagangan berlangsung di atas jukung di Sungai Martapura. Cocok untuk wisata sungai, budaya, dan kuliner pagi. Jadwal dan transportasi harus diverifikasi.","source":"https://dispar.kalselprov.go.id/tinjau-galeri-di-lok-baintan-langkah-kuatkan-wisata-pasar-terapung/"},\n {"title":"Pulau Kembang","category":"destinasi, alam","text":"Pulau di Sungai Barito yang dikenal sebagai habitat monyet ekor panjang. Pengunjung perlu menjaga barang, tidak memberi pakan sembarangan, dan mengikuti aturan setempat.","source":"https://www.indonesia.travel/id/id/destination/kalimantan/south-kalimantan"},\n {"title":"Loksado dan Pegunungan Meratus","category":"destinasi, alam, budaya","text":"Kawasan di Hulu Sungai Selatan dengan lanskap Pegunungan Meratus, hutan tropis, dan desa tradisional Dayak Meratus. Aktivitas dapat meliputi pengalaman alam dan budaya. Periksa cuaca, akses, pemandu, dan kondisi jalur.","source":"https://www.indonesia.travel/id/id/destination/kalimantan/south-kalimantan"},\n {"title":"Tahura Sultan Adam","category":"destinasi, alam","text":"Kawasan hutan raya untuk trekking, piknik, dan menikmati alam. Jam kunjung, tiket, serta kondisi jalur harus diperiksa kepada pengelola.","source":"https://www.indonesia.travel/id/id/destination/kalimantan/south-kalimantan"},\n {"title":"Masjid Sultan Suriansyah","category":"destinasi, sejarah, religi","text":"Destinasi religi dan sejarah di Kuin Utara, Banjarmasin, dengan arsitektur Banjar. Wisatawan harus berpakaian sopan dan menghormati waktu ibadah.","source":"https://www.indonesia.travel/id/id/travel-ideas/adventure/7-amazing-places-you-need-to-visit-in-banjarmasin"},\n {"title":"Museum Waja Sampai Kaputing","category":"destinasi, sejarah","text":"Museum perjuangan rakyat Kalimantan Selatan di kawasan Sungai Jingah, Banjarmasin, dan bertempat di rumah tradisional Banjar. Periksa jam buka sebelum berkunjung.","source":"https://www.indonesia.travel/id/id/travel-ideas/adventure/7-amazing-places-you-need-to-visit-in-banjarmasin"},\n {"title":"Gunung Halau-halau","category":"wisata gunung","text":"Gunung di kawasan Pegunungan Meratus dan dikenal sebagai atap Kalimantan Selatan. Sumber mencantumkan angka ketinggian yang berbeda, sehingga ketinggian, jalur, izin, cuaca, dan status akses harus diverifikasi kepada pihak lokal sebelum pendakian.","source":"https://id.wikipedia.org/wiki/Daftar_gunung_di_Kalimantan_Selatan | https://kalsel.indozone.id/lifestyle/2486141360/6-rekomendasi-gunung-di-kalimantan-selatan-yang-bisa-kamu-rasakan-petualangannya"},\n {"title":"Gunung Bayutawar","category":"wisata gunung","text":"Gunung di Hulu Sungai Tengah. Sumber menyebut medan hutan tropis dan curam. Detail jalur, izin, pemandu, cuaca, dan keselamatan harus diverifikasi.","source":"https://id.wikipedia.org/wiki/Daftar_gunung_di_Kalimantan_Selatan"},\n {"title":"Gunung Kahung","category":"wisata gunung","text":"Tujuan petualangan di wilayah Aranio, Kabupaten Banjar. Akses disebut berkaitan dengan kawasan Waduk Riam Kanan dan Desa Belangian. Verifikasi transportasi air, rute, izin, pemandu, dan status akses.","source":"https://kalsel.indozone.id/lifestyle/2486141360/6-rekomendasi-gunung-di-kalimantan-selatan-yang-bisa-kamu-rasakan-petualangannya"},\n {"title":"Gunung Haurbunak, Pangilingin, Hauk, Lumut, dan Jambangan","category":"wisata gunung","text":"Daftar gunung Kalimantan Selatan mencantumkan Haurbunak dan Pangilingin di Tanah Laut, Hauk di Balangan, Lumut di Tabalong, serta Jambangan di Kotabaru. Sumber tidak merinci kelayakan wisata atau jalur pendakian, jadi jangan menganggap akses terbuka tanpa verifikasi.","source":"https://id.wikipedia.org/wiki/Daftar_gunung_di_Kalimantan_Selatan"},\n {"title":"Budaya Banjar","category":"budaya Banjar","text":"Kebudayaan Kalimantan Selatan berkaitan erat dengan kehidupan sungai. Jukung digunakan di Pasar Terapung Lok Baintan. Sasirangan dikenal sebagai kain tradisional Kalimantan Selatan. Musik tradisi yang disebut Dinas Pariwisata meliputi gamelan Banjar dan musik panting.","source":"https://www.indonesia.travel/id/id/destination/kalimantan/south-kalimantan | https://dispar.kalselprov.go.id/dispar-hadiri-antasan-banjar-ethnogroove/"},\n {"title":"Kuliner Kalimantan Selatan","category":"kuliner","text":"Kuliner yang disebut sumber pariwisata antara lain Soto Banjar, Ketupat Kandangan, ikan bakar berbahan ikan sungai atau tambak, nasi kuning ikan haruan, amplang, dan kue bingka. Soto Banjar memakai beragam rempah. Ketupat Kandangan berasal dari Kandangan dan lazim memakai ikan haruan atau gabus.","source":"https://www.indonesia.travel/id/id/travel-ideas/gastronomy/kuliner-khas-kalimantan-selatan-ini-pantang-dilewatkan"},\n {"title":"Sejarah Banjarmasin","category":"sejarah","text":"Banjarmasin memiliki warisan Kesultanan Banjar dan jaringan sungai yang membentuk perdagangan serta kehidupan masyarakat. Pasar Terapung Lok Baintan disebut telah ada sejak masa Kesultanan Banjar.","source":"https://www.indonesia.travel/id/id/destination/kalimantan/south-kalimantan/banjarmasin | https://dispar.kalselprov.go.id/tinjau-galeri-di-lok-baintan-langkah-kuatkan-wisata-pasar-terapung/"},\n {"title":"Rekomendasi wisata","category":"rekomendasi","text":"Untuk wisata sungai dan budaya, pertimbangkan Pasar Terapung Lok Baintan serta Banjarmasin. Untuk alam dan petualangan, pertimbangkan Meratus, Loksado, dan Tahura Sultan Adam. Untuk sejarah dan religi, pertimbangkan Masjid Sultan Suriansyah dan Museum Waja Sampai Kaputing. Sesuaikan dengan minat, durasi, kondisi fisik, dan informasi operasional terbaru.","source":"https://www.indonesia.travel/id/id/destination/kalimantan/south-kalimantan"},\n {"title":"Panduan itinerary","category":"itinerary","text":"Itinerary perlu disusun berdasarkan durasi, titik awal, minat, mobilitas, dan waktu operasional yang telah diverifikasi. Ide satu hari: pagi wisata sungai atau pasar terapung, siang kuliner Banjar, sore wisata sejarah atau tepian sungai. Ini contoh rancangan, bukan jadwal resmi.","source":"Saran aplikasi; verifikasi semua informasi operasional"},\n {"title":"FAQ destinasi","category":"tanya jawab destinasi","text":"Jawaban destinasi sebaiknya menjelaskan daya tarik, wilayah, aktivitas, dan catatan verifikasi. Jangan mengarang harga, jam buka, status jalur, cuaca, waktu tempuh, atau aturan terbaru.","source":"Kebijakan jawaban aplikasi"}\n]\n\n@st.cache_resource\ndef get_embedder():\n    return SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")\n\n@st.cache_resource\ndef get_index():\n    texts=[f"{d[\'title\']}. Kategori: {d[\'category\']}. {d[\'text\']}" for d in RAG_DATA]\n    vectors=np.asarray(get_embedder().encode(texts,normalize_embeddings=True),dtype="float32")\n    index=faiss.IndexFlatIP(vectors.shape[1]); index.add(vectors)\n    return index\n\ndef retrieve(question,k=4):\n    q=np.asarray(get_embedder().encode([question],normalize_embeddings=True),dtype="float32")\n    scores,ids=get_index().search(q,min(k,len(RAG_DATA)))\n    return [{**RAG_DATA[i],"score":float(s)} for s,i in zip(scores[0],ids[0]) if i>=0]\n\ndef load_memory():\n    try:\n        value=json.loads(Path(MEMORY_FILE).read_text(encoding="utf-8"))\n        return value if isinstance(value,list) else []\n    except Exception: return []\n\ndef save_memory(messages):\n    Path(MEMORY_FILE).write_text(json.dumps(messages,ensure_ascii=False,indent=2),encoding="utf-8")\n\ndef question_key(text):\n    normalized=re.sub(r"[^a-z0-9 ]+"," ",text.lower())\n    normalized=re.sub(r"\\s+"," ",normalized).strip()\n    return hashlib.sha256(normalized.encode()).hexdigest()\n\ndef prior_answer(question,messages):\n    key=question_key(question)\n    for i in range(len(messages)-1):\n        if messages[i].get("role")=="user" and question_key(messages[i].get("content",""))==key and messages[i+1].get("role")=="assistant":\n            return messages[i+1]\n    return None\n\ndef groq_answer(messages,key):\n    response=Groq(api_key=key).chat.completions.create(model=MODEL_NAME,messages=messages,temperature=0.2,max_completion_tokens=1400)\n    return response.choices[0].message.content\n\ndef answer(question,docs,key,threshold):\n    best=docs[0]["score"] if docs else -1\n    history="\\n".join(f"{m[\'role\']}: {m[\'content\']}" for m in st.session_state.messages[-8:])\n    if best>=threshold:\n        context="\\n\\n".join(f"[S{i}] {d[\'title\']}\\n{d[\'text\']}\\nSumber: {d[\'source\']}" for i,d in enumerate(docs,1))\n        prompt=f"""Anda adalah asisten wisata Kalimantan Selatan. Jawab pertanyaan dengan konteks RAG berikut dan gunakan sitasi [S1], [S2].\nJangan mengarang harga, jam buka, status jalur, cuaca, waktu tempuh, atau aturan terbaru. Untuk itinerary, sebutkan bahwa susunan adalah saran dan perlu verifikasi operasional.\nRIWAYAT:\\n{history}\\nKONTEKS:\\n{context}\\nPERTANYAAN: {question}"""\n        return groq_answer([{"role":"user","content":prompt}],key),"RAG + Qwen3.8",best\n    prompt=f"""Jawab pertanyaan berikut menggunakan kemampuan umum model qwen/qwen3.8-27b karena RAG tidak menemukan konteks yang cukup relevan: {question}\nKonteks percakapan singkat:\\n{history}\nNyatakan secara singkat bahwa jawaban berasal dari pengetahuan model, bukan RAG atau pencarian web langsung. Jangan mengarang informasi terkini; sarankan verifikasi resmi bila diperlukan."""\n    return groq_answer([{"role":"user","content":prompt}],key),"Fallback langsung ke qwen/qwen3.8-27b",best\n\nst.title("🌿 Wisata Kalsel AI")\nst.caption("Rekomendasi, itinerary, gunung, budaya Banjar, kuliner, sejarah, dan tanya jawab destinasi")\nkey=os.getenv("GROQ_API_KEY","")\nwith st.sidebar:\n    st.success(f"LLM: Groq · {MODEL_NAME}")\n    top_k=st.slider("Top-K RAG",2,8,4)\n    threshold=st.slider("Ambang relevansi RAG",0.15,0.80,0.42,0.01)\n    st.info("Jika skor RAG di bawah ambang, pertanyaan diarahkan langsung ke Qwen3.8.")\n    if not key: st.warning("GROQ_API_KEY belum dimuat dari Colab Secrets.")\n    if st.button("Hapus percakapan dan memory"):\n        st.session_state.messages=[]; Path(MEMORY_FILE).unlink(missing_ok=True); st.rerun()\n    if Path(MEMORY_FILE).exists():\n        st.download_button("Unduh memory",Path(MEMORY_FILE).read_bytes(),"wisata_kalsel_chat_memory.json","application/json")\n\nif "messages" not in st.session_state:\n    st.session_state.messages=load_memory() or [{"role":"assistant","content":"Halo! Pilih jenis bantuan atau tulis pertanyaan tentang wisata Kalimantan Selatan."}]\n\nst.subheader("🧭 Pilih jenis bantuan")\noptions={\n "Rekomendasi wisata":"Rekomendasikan wisata Kalimantan Selatan dan tanyakan minat serta durasi saya.",\n "Buat itinerary":"Bantu buat itinerary Kalimantan Selatan. Tanyakan durasi, titik awal, minat, dan kebutuhan saya.",\n "Wisata gunung":"Jelaskan pilihan wisata gunung di Kalimantan Selatan dari RAG.",\n "Budaya Banjar":"Ceritakan budaya Banjar yang relevan bagi wisatawan.",\n "Kuliner":"Rekomendasikan kuliner khas Kalimantan Selatan dan jelaskan cirinya.",\n "Sejarah":"Jelaskan wisata sejarah Kalimantan Selatan dari RAG.",\n "Tanya jawab destinasi":"Tanyakan destinasi Kalimantan Selatan yang ingin saya ketahui."\n}\ncols=st.columns(4)\nfor i,(label,prompt) in enumerate(options.items()):\n    if cols[i%4].button(label,use_container_width=True): st.session_state.quick_prompt=prompt\n\nfor m in st.session_state.messages:\n    with st.chat_message(m["role"]):\n        st.markdown(m["content"])\n        if m.get("meta"): st.caption(m["meta"])\n\ntyped=st.chat_input("Tanyakan wisata Kalimantan Selatan...")\nq=typed or st.session_state.pop("quick_prompt",None)\nif q:\n    cached=prior_answer(q,st.session_state.messages)\n    st.session_state.messages.append({"role":"user","content":q})\n    with st.chat_message("user"): st.markdown(q)\n    with st.chat_message("assistant"):\n        if cached:\n            text=cached["content"]; meta="Jawaban diambil dari memory karena pertanyaan identik pernah diajukan."\n            st.markdown(text); st.caption(meta)\n        elif not key:\n            text="GROQ_API_KEY belum tersedia dari Colab Secrets."; meta=""; st.error(text)\n        else:\n            docs=retrieve(q,top_k)\n            try:\n                with st.spinner("Mencari jawaban..."): text,source_mode,best=answer(q,docs,key,threshold)\n                meta=f"Mode: {source_mode} | skor RAG terbaik: {best:.3f}"\n                st.markdown(text); st.caption(meta)\n                with st.expander("Hasil retrieval"):\n                    for d in docs: st.write(f"{d[\'title\']} | {d[\'score\']:.3f} | {d[\'source\']}")\n            except Exception as exc:\n                text=f"Permintaan gagal: {exc}"; meta=""; st.error(text)\n    st.session_state.messages.append({"role":"assistant","content":text,"meta":meta})\n    save_memory(st.session_state.messages)\n'
Path('/content/app.py').write_text(app_code,encoding='utf-8')
print('app.py dibuat')


app.py dibuat


**Jalankan Streamlit**

In [17]:
proc = run_streamlit("app.py")

Streamlit berjalan: NgrokTunnel: "https://everglade-huntress-flight.ngrok-free.dev" -> "http://localhost:8501"


**Stop Streamlit**

In [ ]:
# Hentikan app sebelumnya sebelum menjalankan app baru
import os, signal
try:
    proc.terminate()
    print("App dihentikan.")
except:
    pass